# LoveDA C3: INFO policy search only

Notebook nay chi tim policy cho Exact-25% Adaptive Spectral Mamba. No khong train final va khong doc official LoveDA Val. Hay chon accelerator **GPU T4 x2**. Ket qua can luu la `info_policy_artifact.zip`.

Neu Kaggle bao `ERRORED_MOUNTING_DATASET` truoc khi notebook chay, hay go dataset khoi Input. Cell du lieu se thu tai public dataset bang Kaggle API.


In [ ]:
import json, os, shutil, subprocess, sys, torch
from pathlib import Path

REPO_URL = "https://github.com/TuanHungGiang/LandCover.git"
WORK = Path("/kaggle/working/LandCover")

if (WORK / ".git").is_dir():
    subprocess.run(["git", "-C", str(WORK), "pull", "--ff-only"], check=True)
else:
    shutil.rmtree(WORK, ignore_errors=True)
    subprocess.run(["git", "clone", REPO_URL, str(WORK)], check=True)

os.chdir(WORK)
print(subprocess.check_output(["git", "log", "-1", "--oneline"], text=True).strip())
print("GPUs:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.device_count() >= 2, "Select Kaggle accelerator GPU T4 x2"


In [ ]:
!pip -q install pytorch-lightning==2.0.6 torchmetrics==0.11.4 timm==0.6.7 einops addict prettytable ttach safetensors
!python -c "import torch, pytorch_lightning, timm; print(torch.__version__, pytorch_lightning.__version__, timm.__version__)"


In [ ]:
# Prefer an attached dataset, but download it if no valid split is visible.
DATA_INPUT = Path("/kaggle/input")
DOWNLOAD_IF_MISSING = True

def find_loveda_split(base, split):
    matches = []
    for root, dirs, _ in os.walk(base):
        if Path(root).name.lower() == split.lower() and {"Urban", "Rural"}.issubset(dirs):
            matches.append(Path(root))
    if not matches:
        raise FileNotFoundError(f"Cannot find LoveDA/{split} below {base}")
    return min(matches, key=lambda p: len(str(p)))

try:
    find_loveda_split(DATA_INPUT, "Train")
except FileNotFoundError:
    if not DOWNLOAD_IF_MISSING:
        raise
    DATA_INPUT = Path("/kaggle/working/loveda_download")
    DATA_INPUT.mkdir(parents=True, exist_ok=True)
    subprocess.run(["kaggle", "datasets", "download",
                    "-d", "alienxc137/loveda-satellite-images-semantic-segmentation",
                    "-p", str(DATA_INPUT), "--unzip"], check=True)

# Policy search uses only LoveDA Train. Official Val is neither found nor copied.
# Train from local working storage, not through the long-lived Kaggle mount.
data_root = WORK / "data" / "2021LoveDA"
data_root.mkdir(parents=True, exist_ok=True)
src = find_loveda_split(DATA_INPUT, "Train")
dst = data_root / "Train"
if dst.is_symlink():
    dst.unlink()
if not dst.exists():
    print(f"Copying Train: {src} -> {dst}")
    shutil.copytree(src, dst)
total = 0
for domain in ("Urban", "Rural"):
    images = {p.stem for p in (dst / domain / "images_png").glob("*.png")}
    masks = {p.stem for p in (dst / domain / "masks_png").glob("*.png")}
    assert images == masks, f"Train/{domain}: image-mask IDs differ"
    total += len(images)
assert total == 2522, f"Train: {total} != 2522"
print(f"Train: paired images={total}, local={dst}")


In [ ]:
# m11 is the recommended lightweight model. Change to m15 only for a separate study.
MODEL = "m11"  # m11 | m15
RUNS = {
    "m11": {
        "config": "configs/loveda/gee_repvit_landcover_adaptive_spectral.py",
        "search_config": "configs/loveda/gee_repvit_landcover_adaptive_spectral_search.py",
        "pretrain": "repvit_m1_1",
    },
    "m15": {
        "config": "configs/loveda/gee_repvit_landcover_adaptive_spectral_m15.py",
        "search_config": "configs/loveda/gee_repvit_landcover_adaptive_spectral_search_m15.py",
        "pretrain": "repvit_m1_5",
    },
}
run = RUNS[MODEL]
for key in ("config", "search_config"):
    assert Path(run[key]).is_file(), run[key]

pretrain_path = Path("pretrain") / f"{run['pretrain']}_distill_450e.pth"
if not pretrain_path.is_file():
    subprocess.run([sys.executable, "tools/convert_repvit_timm.py",
                    "--model", run["pretrain"], "--dst", str(pretrain_path)], check=True)
print(run)
print(pretrain_path, pretrain_path.stat().st_size // (1024 ** 2), "MiB")


In [ ]:
# Fast invariants plus a two-batch data/model smoke test on an internal Train fold.
for check in ("tools/check_info_token_search.py",
              "tools/check_loveda_search_split.py",
              "tools/check_adaptive_spectral_scan.py"):
    subprocess.run([sys.executable, check], check=True)

smoke = [sys.executable, "train.py", "-c", run["search_config"], "--set",
         "gpus=[0]", "epoch=1", "check_val_every_n_epoch=1",
         "limit_train_batches=2", "limit_val_batches=2",
         "dataset_config.train_mode.loader.batch_size=1",
         "dataset_config.val_mode.loader.batch_size=1",
         "accumulate_grad_batches=1", "save_top_k=0", "save_last=False",
         "exp_name=work_dirs/smoke_info_search",
         "optimizer_config.scheduler.max_epoch=1"]
subprocess.run(smoke, check=True)


In [ ]:
# 4 initial policies + 4 INFO children, each on 3 folds, then 2 leaders on 3 folds = 30 jobs.
SEARCH_OUT = Path("work_dirs/info_token_search")
search_cmd = [
    sys.executable, "tools/search_token_policy_info.py",
    "--gpus", "0", "1",
    "--population", "4",
    "--generations", "1",
    "--confirm_top", "2",
    "--cv-penalty", "0.5",
    "--val_fraction", "1.0",  # avoid an Urban-first half-validation bias
    "--config", run["search_config"],
    "--final-config", run["config"],
    "--out", str(SEARCH_OUT),
]
print("SEARCH:", " ".join(search_cmd))
subprocess.run(search_cmd, check=True)


In [ ]:
policy_path = SEARCH_OUT / "best_policy.json"
manifest_path = SEARCH_OUT / "split_manifest.json"
assert policy_path.is_file(), policy_path
assert manifest_path.is_file(), manifest_path

artifact = json.loads(policy_path.read_text(encoding="utf-8"))
assert artifact["exact_ratio"] == 0.25
assert artifact["search_split"]["num_folds"] >= 3
assert artifact["search_split"]["manifest_sha256"]
assert os.path.normpath(artifact["final_config"]) == os.path.normpath(run["config"])
assert "model_config.seghead.scan_cfg.ratio=0.25" in artifact["final_overrides"]

print("Algorithm:", artifact["algorithm"])
print("Selected policy:")
print(json.dumps(artifact["selected"]["policy"], indent=2))
print("CV mIoU:", artifact["selected"]["miou"])
print("CV hard-class IoU:", artifact["selected"]["hard_iou"])
print("Block latency (ms):", artifact["selected"]["latency_ms"])
print("Final overrides:")
print("\n".join(artifact["final_overrides"]))


In [ ]:
# Export only the reproducibility artifact, not the large screening checkpoints.
EXPORT = Path("/kaggle/working/info_policy_artifact")
EXPORT.mkdir(parents=True, exist_ok=True)
shutil.copy2(policy_path, EXPORT / "best_policy.json")
shutil.copy2(manifest_path, EXPORT / "split_manifest.json")

commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
metadata = {
    "git_commit": commit,
    "model": MODEL,
    "config": run["config"],
    "search_config": run["search_config"],
    "algorithm": artifact["algorithm"],
    "exact_ratio": artifact["exact_ratio"],
}
(EXPORT / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
archive = shutil.make_archive("/kaggle/working/info_policy_artifact", "zip", EXPORT)
print("READY:", archive, Path(archive).stat().st_size, "bytes")
print("Choose Save Version, then keep info_policy_artifact.zip for the final-training session.")


## Output

Sau khi cell cuoi ket thuc, chon **Save Version** va giu file `/kaggle/working/info_policy_artifact.zip`. Session train final se dat `RUN_META_SEARCH=False` va nap `best_policy.json` tu artifact nay.
